# 02 · KV-cache decoding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/02_kv_cache.ipynb) &nbsp; 📖 Concept deck: [KV cache](https://nilesh-salpe.github.io/visual-ai/inference/02-kv-cache/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** incremental decoding with a key/value cache that gives exactly the same logits as recomputing the whole prefix, but does only one new token of work per step.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def init_params(vocab=64, d_model=32, n_heads=4, n_layers=2, max_len=64, seed=0):
    """A tiny GPT-2-style model with random weights (pre-norm blocks, tied embeddings)."""
    r = np.random.default_rng(seed)
    n = lambda *s: r.normal(0, 0.2, s)
    layers = [dict(ln1_g=1 + n(d_model) * .1, ln1_b=n(d_model) * .1, Wqkv=n(d_model, 3 * d_model), bqkv=n(3 * d_model) * .1,
                   Wo=n(d_model, d_model), bo=n(d_model) * .1, ln2_g=1 + n(d_model) * .1, ln2_b=n(d_model) * .1,
                   W1=n(d_model, 4 * d_model), b1=n(4 * d_model) * .1, W2=n(4 * d_model, d_model), b2=n(d_model) * .1)
              for _ in range(n_layers)]
    return dict(wte=n(vocab, d_model), wpe=n(max_len, d_model), layers=layers,
                lnf_g=1 + n(d_model) * .1, lnf_b=n(d_model) * .1, n_heads=n_heads)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _ln(x, g, b, eps=1e-5):
    m = x.mean(-1, keepdims=True); v = x.var(-1, keepdims=True)
    return (x - m) / np.sqrt(v + eps) * g + b

def _attn(x, L, H):
    T, d = x.shape; dh = d // H
    q, k, v = np.split(x @ L['Wqkv'] + L['bqkv'], 3, axis=-1)
    q, k, v = [a.reshape(T, H, dh).transpose(1, 0, 2) for a in (q, k, v)]
    s = q @ k.transpose(0, 2, 1) / np.sqrt(dh)
    s = np.where(np.tril(np.ones((T, T), bool)), s, -np.inf)
    o = softmax(s) @ v
    return o.transpose(1, 0, 2).reshape(T, d) @ L['Wo'] + L['bo']

def _gpt(P, tokens):
    T = len(tokens); x = P['wte'][tokens] + P['wpe'][:T]
    for L in P['layers']:
        x = x + _attn(_ln(x, L['ln1_g'], L['ln1_b']), L, P['n_heads'])
        x = x + gelu(_ln(x, L['ln2_g'], L['ln2_b']) @ L['W1'] + L['b1']) @ L['W2'] + L['b2']
    return _ln(x, P['lnf_g'], P['lnf_b']) @ P['wte'].T

P = init_params()

def new_cache(params):
    d = params['wte'].shape[1]
    return [{'k': np.zeros((0, d)), 'v': np.zeros((0, d))} for _ in params['layers']]

def generate_naive(params, prompt, n_new):
    """Greedy decoding WITHOUT a cache: re-runs the full forward pass for every new token."""
    toks = list(prompt)
    for _ in range(n_new):
        toks.append(int(_gpt(params, toks)[-1].argmax()))
    return toks
print('helpers ready: new_cache(P), generate_naive(P, prompt, n), _gpt(P, tokens) = full forward')

## Task 1: One decoding step with a cache

Process **one** token at position `pos`:
- $x = W_{te}[\text{token}] + W_{pe}[\text{pos}]$ (shape [d]).
- For each layer: compute this token's `q, k, v` from $\mathrm{LN}_1(x)$, **append** `k` and `v` to `cache[l]`, then attend with `q` over **all cached** keys and values. No mask is needed, because the cache only holds the past.
- Finish the block as usual (residual, MLP, residual), apply the final LN, and return logits of shape [vocab].

`cache[l]['k']` and `cache[l]['v']` have shape [t, d], with heads still concatenated. Split them into heads inside attention.

In [ ]:
def forward_step(params, token, pos, cache):
    """Returns logits [vocab] for `token` at `pos`, appending this token's k, v to `cache` (mutated in place)."""
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_forward_step(f):
    toks = [5, 9, 1, 33, 7, 21]
    full = _gpt(P, toks)
    cache = new_cache(P)
    for t, tok in enumerate(toks):
        out = f(P, tok, t, cache)
        assert out.shape == (64,), f"logits shape {out.shape} != (64,)"
        assert np.allclose(out, full[t], atol=1e-5), f"step {t}: cached logits differ from the full forward pass"
    assert cache[0]['k'].shape == (6, 32), f"cache should hold 6 keys of width 32, got {cache[0]['k'].shape}"

check("forward_step", _test_forward_step, forward_step)

## Task 2: Greedy generation with the cache

Feed the prompt token by token (this is the **prefill**), then repeatedly take the argmax of the last logits, append it, and feed it back (this is **decode**). The output must equal `generate_naive`.

In [ ]:
def generate_cached(params, prompt, n_new, step=None):
    """Greedy decoding with a KV cache. `step` defaults to your forward_step."""
    step = step or forward_step
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_generate_cached(f):
    prompt = [4, 8, 15, 16]
    assert f(P, prompt, 12) == generate_naive(P, prompt, 12), "tokens differ from naive greedy decoding"

check("generate_cached", _test_generate_cached, generate_cached)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: One decoding step with a cache
def forward_step_solution(params, token, pos, cache):
    H = params['n_heads']; d = params['wte'].shape[1]; dh = d // H
    x = params['wte'][token] + params['wpe'][pos]
    for L, c in zip(params['layers'], cache):
        h = _ln(x[None], L['ln1_g'], L['ln1_b'])[0]
        q, k, v = np.split(h @ L['Wqkv'] + L['bqkv'], 3)
        c['k'] = np.vstack([c['k'], k]); c['v'] = np.vstack([c['v'], v])
        t = c['k'].shape[0]
        Q = q.reshape(H, 1, dh)
        K = c['k'].reshape(t, H, dh).transpose(1, 0, 2)
        V = c['v'].reshape(t, H, dh).transpose(1, 0, 2)
        o = softmax(Q @ K.transpose(0, 2, 1) / np.sqrt(dh)) @ V          # [H, 1, dh]
        x = x + o.reshape(d) @ L['Wo'] + L['bo']
        h2 = _ln(x[None], L['ln2_g'], L['ln2_b'])[0]
        x = x + gelu(h2 @ L['W1'] + L['b1']) @ L['W2'] + L['b2']
    return _ln(x[None], params['lnf_g'], params['lnf_b'])[0] @ params['wte'].T

check("forward_step (reference)", _test_forward_step, forward_step_solution)

In [ ]:
# @title Solution for Task 2: Greedy generation with the cache
def generate_cached_solution(params, prompt, n_new, step=None):
    step = step or forward_step_solution
    cache, toks = new_cache(params), list(prompt)
    for pos, tok in enumerate(toks):          # prefill
        logits = step(params, tok, pos, cache)
    for _ in range(n_new):                     # decode
        nxt = int(logits.argmax()); toks.append(nxt)
        logits = step(params, nxt, len(toks) - 1, cache)
    return toks

check("generate_cached (reference)", _test_generate_cached, generate_cached_solution)

---
## Experiments
These use the reference solutions, so they run even before you finish.

In [ ]:
# Work per generated token: naive recomputes the prefix, cached does one token.
P2 = init_params(max_len=512)
prompt = list(range(1, 33))
for n in (32, 96, 192):
    t0 = time.perf_counter(); generate_naive(P2, prompt, n); t_naive = time.perf_counter() - t0
    t0 = time.perf_counter(); generate_cached_solution(P2, prompt, n); t_cache = time.perf_counter() - t0
    naive_rows = sum(len(prompt) + i for i in range(n))     # token rows processed by the naive loop
    print(f"{n:4d} new tokens: naive {t_naive*1e3:7.1f} ms ({naive_rows} token-rows)   cached {t_cache*1e3:7.1f} ms ({len(prompt)+n} token-rows)")

---
## Takeaways
- The cache stores each layer's **K and V** for every past token. A new token computes only its own q, k and v and attends over the cache.
- Logits are **identical** to recomputing everything, so the cache is an exact optimization.
- The cost moves from compute to **memory**: the cache grows linearly with context length and batch size (notebook 04).